## Лабораторная работа №1

Цель: написание алгоритма knn с нуля для классификации изображений MNIST (использовать Евклидово расстояние)
1) Классификация на основе ближайшего соседа;
2) Ускорение программного кода на Python с помощью Numba и Numpy

In [11]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from timeit_decorator import timeit
X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                    as_frame=False, parser="auto")
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=10000, test_size=500, random_state=0)

In [ ]:
# Евклидово расстояние вычисляется по формуле: sqrt(sum((ai-bi)^2))
# Вычисление корня на сотнях миллионов элементов стоит большого времени, а нам нужен лишь порядок ближайших соседей
from collections import Counter
from numba import njit, prange
import numpy as np

def test_accuracy(y_test, preds):
    _ = np.zeros_like(y_test)
    for i in range(len(y_test)):
        _[i] = (y_test[i]==preds[i])
    return _.mean()


@timeit
def compute_distances_classic(X_test, X_train):
    dist_array = []
    for x in X_test:
        row = []
        for elem in X_train:
            temp = sum((x-elem)**2)
            row.append(temp)
        dist_array.append(row)
    return dist_array


# Можно ускорить, если сделать чисто матричные операции без циклов.
@timeit
def compute_distances_np(X_test, X_train):
    dist_array = np.zeros([500, 10000])
    for i,x in enumerate(X_test):
        for j,elem in enumerate(X_train):
            temp = ((x-elem)**2).sum()
            dist_array[i][j] = temp
    return dist_array


@timeit
@njit(cache=True)
def compute_distances_numba(X_test, X_train):
    n_test, n_feat = X_test.shape
    n_train = X_train.shape[0]
    dists = np.empty((n_test, n_train), dtype=np.float64)
    for i in range(n_test):
        for j in range(n_train):
            s = 0.0
            for f in range(n_feat):
                d = X_test[i, f] - X_train[j, f]
                s += d * d
            dists[i, j] = s
    return dists


@timeit
def predict_classic(dists, y_train, k):
    preds = []
    for i in range(len(dists)):
        idx = sorted(range(len(dists[i])), key=dists[i].__getitem__)[:k]
        labels = y_train[idx]
        preds.append(Counter(labels).most_common(1)[0][0])
    return preds


@timeit
def predict_np(dists, y_train, k):
    preds = np.empty(len(dists), dtype=int)
    for i in range(len(dists)):
        idx = np.argpartition(dists[i], k)[:k]
        labels = y_train[idx]
        preds[i] = np.bincount(labels, minlength=10).argmax()
    return preds


@timeit
@njit(cache=True)
def predict_numba(dists, y_train, k):
    n_test = dists.shape[0]
    preds = np.empty(n_test, dtype=np.int64)
    for i in range(n_test):
        idx = np.argsort(dists[i])[:k]
        counts = np.zeros(10, dtype=np.int64)
        for t in range(k):
            counts[y_train[idx[t]]] += 1
        preds[i] = counts.argmax()
    return preds


k = int(input())

In [20]:
classic_dists = compute_distances_classic(X_test, X_train)
classic_preds = predict_classic(classic_dists, y_train, k)

print(f"Classic Python accuracy: {test_accuracy(y_test, classic_preds)}")

2026-09-20 15:21:18 [INFO] (timeit.decorator) compute_distances_classic: Exec: 340.373s
2026-09-20 15:21:20 [INFO] (timeit.decorator) predict_classic: Exec: 1.558s


Classic Python accuracy: 0.956


In [18]:
numpy_dists = compute_distances_np(X_test, X_train)
numpy_preds = predict_np(numpy_dists, y_train, k)

print(f"Numpy accuracy: {test_accuracy(y_test, numpy_preds)}")

2026-09-20 15:12:09 [INFO] (timeit.decorator) compute_distances_np: Exec: 26.639s
2026-09-20 15:12:10 [INFO] (timeit.decorator) predict_np: Exec: 21.59ms


Numpy accuracy: 0.954


In [19]:
numba_dists = compute_distances_numba(X_test, X_train)
numba_preds = predict_numba(numba_dists, y_train, k)

print(f"Numba accuracy: {test_accuracy(y_test, numba_preds)}")

2026-09-20 15:12:36 [INFO] (timeit.decorator) compute_distances_numba: Exec: 5.793s
2026-09-20 15:12:36 [INFO] (timeit.decorator) predict_numba: Exec: 811.01ms


Numba accuracy: 0.954


При train_size=10000, test_size=500 и k=3 были получены следующие результаты замеров длительности выполнения алгоритмов:
 * Classic Python: 340.373 секунды
 * Numpy: 26.639 секунды
 * Numba: 5.793 секунды

Классический алгоритм имеет точность 0.956, когда два других 0.954. Я списываю это на неопределенность в разрешении ничьих при выведении предсказания по самой частой метке. Математически все три алгоритма одинаковы.